# FT-Transformer final: teste e modelo para o Hugging Face

**Versão adotada: `f7_triagem`.** É o `o5_top3` (1 bloco, d_token 192, 4 cabeças, dropout de atenção ~0,49, sem oversampling) com o pacote de cinco features da análise de erro: `cobranca_razao`, `antigo`, `caro_sem_backup`, `fibra_sem_protecao` e `atrito_pagamento`. Teve o **maior KS do placar entre todas as versões do FT: 0,5573**. A média das 4 versões de maior KS nas divisões do Optuna ficou em 0,5557 e não entrou.

## O que este notebook faz

1. **Treino final**, como o do MLP: em cada uma das 5 sementes, 20% do treino oficial decidem as épocas pela parada da configuração. Depois a rede retreina no treino inteiro por essas épocas e prevê o teste. A previsão é a média das 5 redes.
2. **Teste, medido uma única vez.** KS com IC95 por bootstrap, AUROC e métricas de classe com o limiar do KS escolhido na OOF do placar (nunca no teste). Comparação com o MLP adotado no mesmo teste, só como conferência.
3. **Pasta `modelo_hf/`** pronta para o Hugging Face: os pesos das 5 redes, `config.json` (colunas, scaler, limiar, métricas), `churn_ft.py` (pré-processamento e modelo, sem depender do projeto) e o `README.md` do modelo. O notebook confere que o pacote, carregado sozinho, prevê o mesmo que o treino.
4. **Envio ao Hugging Face (opcional):** com `HF_REPO` preenchido e o secret `HF_TOKEN`, a pasta sobe para um repositório de modelo (privado por padrão).

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/ft_transformer/ft_transformer_final.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN`, `WANDB_API_KEY` e, para enviar ao Hugging Face, `HF_TOKEN` (um token de escrita, criado em huggingface.co → Settings → Access Tokens). **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True` (sem enviar ao Hugging Face). Depois, com `False`, use *Save Version → Save & Run All (Commit)*. São poucos minutos.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/ft-transformer"           # branch com o código (precisa estar publicada no GitHub)
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "ft-final-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"   # vazio = procurar no repositório e em /kaggle/input
POR_GPU = 2                      # processos (dobras ao mesmo tempo) em cada GPU; sem GPU, processos na CPU
FASE7_RUN = "ft-fase7-20261120-0429"   # execução da fase 7 na branch de resultados (ou uma pasta local): traz o placar do FT
ROTULO = "f7_triagem"            # a versão adotada (maior KS do placar do FT)
N_SEMENTES = 5                   # redes do treino final (a previsão é a média)
HF_REPO = ""                     # ex.: "seu-usuario/ft-transformer-churn"; vazio = não envia ao Hugging Face
HF_PRIVADO = True                # o repositório de modelo nasce privado; dá para abrir depois no site
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa configurações novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "ft_transformer" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

PASTA_FT = REPO_DIR / "experimentos" / "ft_transformer"
sys.path[:0] = [str(REPO_DIR), str(PASTA_FT)]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(PASTA_FT), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("ft-final-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"          # antes de importar modelo.py, que lê o limite na importação
    N_SEMENTES, HF_REPO = 2, ""
    print("MODO_TESTE: 3 épocas, 2 sementes e nada vai ao Hugging Face. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import torch
import comum, modelo as M, fase1 as F1, fase2 as F2, fase4 as F4, fase5 as F5, fase6 as F6, fase7 as F7, final as FIN

DISPOSITIVOS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
if DISPOSITIVOS == ["cpu"]:
    POR_GPU = os.cpu_count() or 1                 # sem GPU: um processo por núcleo
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | dispositivos: {DISPOSITIVOS} x {POR_GPU} processos | até {M.MAX_EPOCAS} épocas")

# O envio ao GitHub é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="ft_transformer", config=dict(modelo="ft_transformer", fase="final", dispositivos=DISPOSITIVOS,
                                                               por_gpu=POR_GPU, teste=MODO_TESTE, padrao=M.PADRAO))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def ao_registrar(r):
    """Cada configuração medida vai para o W&B e para o GitHub na hora: se o Kaggle cair, nada se perde."""
    if wandb_run is not None:
        v = r.get("vs_trial0") or {}
        wandb_run.log(dict(rotulo=r["rotulo"], ks_placar=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"],
                           mse_treino=r.get("mse_treino"), mse_val=r.get("mse_val"), epocas=r.get("epocas"),
                           minutos=r.get("tempo_min"), delta_vs_trial0=v.get("delta")))
    salvar()


# As linhas das fases 1 a 7 (com a versão adotada e a OOF dela) entram aqui sem treinar de novo
import tempfile
if FASE7_RUN and not os.path.isfile(os.path.join(PASTA, "placar.jsonl")):
    with tempfile.TemporaryDirectory() as _tmp:
        ok = (shutil.copytree(FASE7_RUN, _tmp, dirs_exist_ok=True) and True) if os.path.isdir(FASE7_RUN) else \
            fluxo.baixar_github(_tmp, FASE7_RUN, RESULTS_REPO_URL or REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
        if ok and os.path.isfile(os.path.join(_tmp, "placar.jsonl")):
            shutil.copy(os.path.join(_tmp, "placar.jsonl"), PASTA)
            shutil.copytree(os.path.join(_tmp, "placar_oof"), os.path.join(PASTA, "placar_oof"), dirs_exist_ok=True)
anteriores = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"}
print(f"Fases 1 a 7: {len(anteriores)} linhas trazidas de {FASE7_RUN}")
assert ROTULO in anteriores, f"{ROTULO} não está no placar trazido de {FASE7_RUN}"


def medir(itens):
    return F1.garantir(PASTA, itens, DISPOSITIVOS, POR_GPU, continuar=continuar, ao_registrar=ao_registrar)


def mostrar(linhas, campos=("vs_trial0", "vs_base")):
    display(F1.tabela(linhas, campos).round(4))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F1.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
Xtr_, _, ytr_, _ = F1._dados()
Xa_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5])
n_param = sum(p.numel() for p in M.FTTransformer(Xa_.shape[1], **M.arquitetura(M.PADRAO)).parameters())
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões sem sobreposição | assinatura do treino {F1.assinatura()}")
print(f"{Xa_.shape[1]} features = {Xa_.shape[1]} tokens + [CLS] | FT-Transformer padrão: {n_param:,} parâmetros para {len(Xtr_):,} clientes")
Xf_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5], features=F4.TRES)
print(f"com as features do MLP: {Xf_.shape[1]} tokens + [CLS] ({', '.join(F4.TRES)})")
for nome, hp in (("tokens categóricos", dict(tokens="cat")), ("PLR", dict(num_emb="plr"))):
    g_, n_ = M.estrutura(hp, list(Xf_.columns), np.asarray(Xf_, dtype="float32"))
    print(f"{nome}: {len(g_)} tokens | colunas com PLR: {[Xf_.columns[j] for j in n_]}")

# 1. Treino final e teste (uma vez)

Se a pasta `modelo_hf/` já existir nesta execução (retomada), nada é treinado de novo.

In [ ]:
linha = anteriores[ROTULO]
c = linha["config"]
print(f"{ROTULO}: KS do placar {linha['ks_kfold']:.4f} | KS OOF {linha['ks_oof']:.4f}")
print(json.dumps(c, sort_keys=True))
DEST = os.path.join(PASTA, "modelo_hf")
arq_fin = os.path.join(PASTA, "ft_final.json")
if os.path.isfile(os.path.join(DEST, "config.json")) and os.path.isfile(arq_fin):
    fin_json = json.load(open(arq_fin))
    met, cfg = fin_json["metricas"], json.load(open(os.path.join(DEST, "config.json")))
    print("retomada: o treino final já foi feito nesta execução")
else:
    t0 = time.time()
    fin = FIN.final(c, N_SEMENTES, DISPOSITIVOS[0])
    print(f"{(time.time() - t0) / 60:.1f} min | épocas por semente: {fin['epocas']}")
    met = FIN.metricas_teste(np.load(os.path.join(PASTA, linha["oof"])), fin)
    comum.registrar_teste_placar(PASTA, "ft_transformer", ROTULO, fin["y_teste"], fin["p_sementes"])
    cfg = FIN.exportar(DEST, ROTULO, c, fin, met, linha)
    fin_json = dict(rotulo=ROTULO, metricas=met, epocas=fin["epocas"], p_sementes=fin["p_sementes"], y_teste=fin["y_teste"])
    json.dump(fin_json, open(arq_fin, "w"), indent=1)
    salvar()

# O pacote exportado, carregado sozinho, prevê o mesmo que o treino
import importlib.util
spec = importlib.util.spec_from_file_location("churn_ft_exportado", os.path.join(DEST, "churn_ft.py"))
pacote = importlib.util.module_from_spec(spec); spec.loader.exec_module(pacote)
_, Xte_, _, _ = F1._dados()
p_pacote = pacote.carregar(DEST).prever(pd.read_csv(os.environ["CHURN_CSV"]).loc[Xte_.index])
assert np.allclose(p_pacote, np.mean(fin_json["p_sementes"], 0), atol=1e-4), "o pacote exportado não prevê o mesmo que o treino"
shutil.rmtree(os.path.join(DEST, "__pycache__"), ignore_errors=True)   # o import da conferência não vai para o Hugging Face
print("modelo_hf/: o pacote exportado prevê o mesmo que o treino |", sorted(os.listdir(DEST)))

print(f"\nTESTE ({ROTULO}): KS {met['ks']:.4f} (IC95 {met['ks_ic95'][0]:.3f} a {met['ks_ic95'][1]:.3f}) | AUROC {met['auroc']:.4f}")
print(f"limiar do KS (da OOF) {met['limiar']:.4f} | recall {met['recall']:.3f} | precisão {met['precision']:.3f} | F1 {met['f1']:.3f}")
print("KS por semente:", [round(k, 4) for k in met["ks_por_semente"]], "| matriz de confusão:", met["confusao"])

# Conferência contra o MLP adotado, no mesmo teste
RES = os.path.join(REPO_DIR, "resultados")
rot_mlp = json.load(open(os.path.join(RES, "escolhidos.json")))["mlp"]["rotulo"]
mlp = {r["rotulo"]: r for r in comum.ler_placar(RES) if r["modelo"] == "mlp"}[rot_mlp]
arqs = [os.path.join(dp, f) for dp, _, fs in os.walk(os.path.join(RES, "mlp")) for f in fs if f.endswith("_final.json")]
fm = next((json.load(open(a)) for a in arqs if json.load(open(a)).get("rotulo") == rot_mlp), None)
if fm:
    yte = np.asarray(fin_json["y_teste"]); pf = np.mean(fin_json["p_sementes"], 0); pm = np.mean(fm["p_sementes"], 0)
    rng, ds = np.random.default_rng(0), []
    for _ in range(2000):
        i = rng.integers(0, len(yte), len(yte)); ds.append(comum.ks(yte[i], pf[i]) - comum.ks(yte[i], pm[i]))
    lo, hi = np.percentile(ds, [2.5, 97.5])
    print(f"\nMLP adotado ({rot_mlp}): KS do teste {mlp['ks_teste']:.4f} | FT - MLP no teste: {comum.ks(yte, pf) - comum.ks(yte, pm):+.4f} [{lo:+.4f}, {hi:+.4f}]")
if wandb_run is not None:
    wandb_run.summary.update(dict(rotulo=ROTULO, ks_teste=met["ks"], auroc_teste=met["auroc"]))
_ = salvar()

# 2. Enviar ao Hugging Face (opcional)

Com `HF_REPO` preenchido e o secret `HF_TOKEN`, a pasta `modelo_hf/` sobe para um repositório de modelo. Ele nasce **privado** (`HF_PRIVADO = True`); para abrir ao público, use *Settings → Change model visibility* no site. Sem `HF_REPO`, baixe a pasta `modelo_hf/` da aba *Output* (ou da branch `resultados` no GitHub) e envie pelo site: *New model → Files → Add file → Upload files*.

In [ ]:
if HF_REPO and not MODO_TESTE:
    hf_token = _secret("HF_TOKEN")
    assert hf_token, "falta o secret HF_TOKEN (token de escrita do Hugging Face)"
    try:
        import huggingface_hub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        import huggingface_hub
    api = huggingface_hub.HfApi(token=hf_token)
    api.create_repo(HF_REPO, repo_type="model", private=HF_PRIVADO, exist_ok=True)
    api.upload_folder(folder_path=DEST, repo_id=HF_REPO, repo_type="model", commit_message=f"FT-Transformer {ROTULO}: KS do teste {met['ks']:.4f}",
                      ignore_patterns=["__pycache__/*", "*.pyc"])
    print(f"Enviado: https://huggingface.co/{HF_REPO}")
else:
    print("Hugging Face: nada enviado (HF_REPO vazio ou MODO_TESTE). A pasta pronta está em", DEST)
open(os.path.join(PASTA, "PLACAR.md"), "w", encoding="utf-8").write(comum.relatorio_placar(PASTA))
salvar()
if wandb_run is not None:
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 60:.1f} min")
print(sorted(os.listdir(PASTA)))

### 📝 Análise

- **KS do teste do FT, e o IC95:** _…_
- **Contra o MLP no mesmo teste:** _…_
- **O KS do teste ficou perto do KS do placar?** _…_